In [2]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import joblib

In [3]:
PROJECT_ROOT = Path("..").resolve()

TRAIN_PATH = PROJECT_ROOT / "data" / "raw" / "train.csv"

df = pd.read_csv(TRAIN_PATH)

df["date"] = pd.to_datetime(df["date"])

print("Shape:", df.shape)
print("Stores:", df["store"].nunique())
print("Date range:", df["date"].min(), "to", df["date"].max())

Shape: (5142, 13)
Stores: 9
Date range: 2021-08-02 00:00:00 to 2023-11-30 00:00:00


In [4]:
store_stats = (
    df.groupby("store")["sales"]
    .agg(["count", "mean", "std", "min", "max"])
    .sort_index()
)

store_stats

,count,mean,std,min,max
store,,,,,
store_0,273,-1.364540,0.162917,-1.633680,-0.733573
store_1,847,-0.032948,0.812338,-1.633680,4.986540
store_2,61,-0.846306,0.284644,-1.283898,-0.110960
store_3,242,-0.700185,0.351627,-1.643008,0.660894
store_4,847,0.188380,0.845076,-1.645340,4.359263
store_5,847,0.169637,0.724249,-1.605698,3.048744
store_6,331,-1.039522,0.388224,-1.656999,1.514364
store_7,847,0.787201,0.890045,-1.624353,7.539953
store_8,847,0.696145,0.972089,-1.610361,6.882362


In [5]:
store_mean = df.groupby("store")["sales"].transform("mean")
store_std = df.groupby("store")["sales"].transform("std")

df["demand_index"] = (
    (df["sales"] - store_mean) /
    store_std.replace(0, 1)
)

In [6]:
def create_date_features(data):

    data = data.copy()

    data["year"] = data["date"].dt.year
    data["month"] = data["date"].dt.month
    data["day"] = data["date"].dt.day

    data["day_of_week"] = data["date"].dt.dayofweek

    data["week_of_year"] = (
        data["date"]
        .dt.isocalendar()
        .week
        .astype(int)
    )

    data["quarter"] = data["date"].dt.quarter

    data["is_weekend"] = (
        data["day_of_week"] >= 5
    ).astype(int)

    return data


df = create_date_features(df)

In [7]:
df = (
    df
    .sort_values(["store", "date"])
    .reset_index(drop=True)
)

df["demand_lag_1"] = (
    df
    .groupby("store")["demand_index"]
    .shift(1)
)

df["demand_lag_7"] = (
    df
    .groupby("store")["demand_index"]
    .shift(7)
)

df["demand_rolling_mean_7"] = (
    df
    .groupby("store")["demand_index"]
    .transform(
        lambda x: x.shift(1).rolling(7).mean()
    )
)

In [8]:
df = df.dropna(
    subset=[
        "demand_lag_1",
        "demand_lag_7",
        "demand_rolling_mean_7"
    ]
).reset_index(drop=True)

print("Processed shape:", df.shape)

Processed shape: (5079, 24)


In [9]:
split_date = pd.Timestamp("2023-06-16")

train_df = df[df["date"] < split_date].copy()
valid_df = df[df["date"] >= split_date].copy()

print("Training:", train_df.shape)
print("Validation:", valid_df.shape)

print(
    "Train:",
    train_df["date"].min(),
    "to",
    train_df["date"].max()
)

print(
    "Validation:",
    valid_df["date"].min(),
    "to",
    valid_df["date"].max()
)

Training: (3682, 24)
Validation: (1397, 24)
Train: 2021-08-09 00:00:00 to 2023-06-15 00:00:00
Validation: 2023-06-16 00:00:00 to 2023-11-30 00:00:00


In [10]:
FEATURES = [
    "is_state_holiday",
    "is_school_holiday",
    "is_special_day",

    "temperature_max",
    "temperature_min",
    "temperature_mean",

    "sunshine_sum",
    "precipitation_sum",

    "year",
    "month",
    "day",
    "day_of_week",
    "week_of_year",
    "quarter",
    "is_weekend",

    "demand_lag_1",
    "demand_lag_7",
    "demand_rolling_mean_7"
]

TARGET = "demand_index"

X_train = train_df[FEATURES]
y_train = train_df[TARGET]

X_valid = valid_df[FEATURES]
y_valid = valid_df[TARGET]

In [11]:
categorical_features = [
    "is_state_holiday",
    "is_school_holiday",
    "is_special_day"
]

numerical_features = [
    feature
    for feature in FEATURES
    if feature not in categorical_features
]

preprocessor = ColumnTransformer(
    transformers=[

        (
            "num",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="median")
                )
            ]),
            numerical_features
        ),

        (
            "cat",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="most_frequent"
                    )
                ),

                (
                    "onehot",
                    OneHotEncoder(
                        handle_unknown="ignore"
                    )
                )
            ]),
            categorical_features
        )
    ]
)

In [12]:
global_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)

global_model.fit(
    X_train,
    y_train
)

print("Global model trained successfully.")

Global model trained successfully.


In [13]:
predictions = global_model.predict(X_valid)

mae = mean_absolute_error(
    y_valid,
    predictions
)

rmse = np.sqrt(
    mean_squared_error(
        y_valid,
        predictions
    )
)

r2 = r2_score(
    y_valid,
    predictions
)

print("=" * 60)
print("GLOBAL MODEL RESULTS")
print("=" * 60)

print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)

GLOBAL MODEL RESULTS
MAE : 0.35609028579202145
RMSE: 0.47601126137307614
R²  : 0.5126385681409231


In [14]:
print("Existing Phase 3 model:")
print("MAE : 0.2686")
print("RMSE:", 0.3475)
print("R²  :", 0.8001)

print("\nNew Global Model:")
print("MAE :", round(mae, 4))
print("RMSE:", round(rmse, 4))
print("R²  :", round(r2, 4))

Existing Phase 3 model:
MAE : 0.2686
RMSE: 0.3475
R²  : 0.8001

New Global Model:
MAE : 0.3561
RMSE: 0.476
R²  : 0.5126


In [15]:
GLOBAL_MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "global_demand_model.joblib"
)

joblib.dump(
    global_model,
    GLOBAL_MODEL_PATH
)

print(
    "Saved:",
    GLOBAL_MODEL_PATH
)

Saved: C:\Users\Menaka\Videos\smart-food-demand-mlops\models\global_demand_model.joblib


In [16]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()

TRAIN_PATH = PROJECT_ROOT / "data" / "raw" / "train.csv"

df = pd.read_csv(TRAIN_PATH)

df["date"] = pd.to_datetime(df["date"])

print("=" * 70)
print("CALIBRATION DATA CHECK")
print("=" * 70)

print("Rows:", len(df))
print("Stores:", df["store"].nunique())

print("\nSales statistics:")
print(df["sales"].describe())

print("\nSales statistics by store:")
print(
    df.groupby("store")["sales"]
      .agg(["count", "mean", "median", "std", "min", "max"])
      .round(4)
)

CALIBRATION DATA CHECK
Rows: 5142
Stores: 9

Sales statistics:
count    5142.000000
mean        0.115530
std         0.997744
min        -1.656999
25%        -0.582000
50%         0.045276
75%         0.712195
max         7.539953
Name: sales, dtype: float64

Sales statistics by store:
         count    mean  median     std     min     max
store                                                 
store_0    273 -1.3645 -1.4052  0.1629 -1.6337 -0.7336
store_1    847 -0.0329 -0.2112  0.8123 -1.6337  4.9865
store_2     61 -0.8463 -0.9248  0.2846 -1.2839 -0.1110
store_3    242 -0.7002 -0.7534  0.3516 -1.6430  0.6609
store_4    847  0.1884  0.0313  0.8451 -1.6453  4.3593
store_5    847  0.1696  0.0756  0.7242 -1.6057  3.0487
store_6    331 -1.0395 -1.1417  0.3882 -1.6570  1.5144
store_7    847  0.7872  0.7309  0.8900 -1.6244  7.5400
store_8    847  0.6961  0.5000  0.9721 -1.6104  6.8824


In [17]:
DEPLOYMENT_STORE = "store_5"

store_df = (
    df[df["store"] == DEPLOYMENT_STORE]
    .sort_values("date")
    .copy()
)

print("=" * 70)
print("DEPLOYMENT STORE CALIBRATION")
print("=" * 70)

print("Store:", DEPLOYMENT_STORE)
print("Rows:", len(store_df))
print("Date range:")
print(store_df["date"].min(), "to", store_df["date"].max())

print("\nSales:")
print(store_df["sales"].describe())

print("\nMedian sales:", store_df["sales"].median())
print("Mean sales:", store_df["sales"].mean())

DEPLOYMENT STORE CALIBRATION
Store: store_5
Rows: 847
Date range:
2021-08-02 00:00:00 to 2023-11-30 00:00:00

Sales:
count    847.000000
mean       0.169637
std        0.724249
min       -1.605698
25%       -0.306838
50%        0.075591
75%        0.601431
max        3.048744
Name: sales, dtype: float64

Median sales: 0.0755909652209577
Mean sales: 0.16963721286255168
